# Ward tiles ("abstract Chicago")

Equal-size ward tiles, plan Phase 4 (`sessions/2026-10-04_plan.md`). Built by
`scripts/build_ward_tiles.py grid` → `data/tables/ward_tiles_grid.geojson`; every setting of the
grid (layouts with every row's start column, which layout is used, aspect, gap, size) is a named
constant at the top of that script. Any ward view can switch to tiles with `MAP_SHAPES = "tiles_grid"` (`scripts/ward_maps.py`).

| § | What | Status |
|---|---|---|
| 1 | Each layout as a table of grid cells | 2026-10-05: south squished + shifted east (user) |
| 2 | Tiles drawn, labeled ward + alder | second draft |
| 3 | Tiles next to the real map; tile → real-ward lines; real east-west position vs grid column | second draft |
| 4 | Real-map neighbors whose N/S or E/W order is flipped on the grid | second draft |
| 5 | Options drawn: layouts, southern start columns, aspect, gap, size fit | for choosing |
| 6 | One existing view on tiles: tenure | demo of `MAP_SHAPES` |
| 7 | Push-apart ("explode") tiles, plan 4b: drawn, compared, options | first draft |

**Latest (user, 2026-10-05):** `GRID_LAYOUT = "plan_2026_10_04"` with 2:1 tiles as the initial
draft; also look at the push-apart tiles (§7) as possibly preferable to a grid.

**Earlier 2026-10-05 changes (user):** tiles at least 1.3 wide per 1 tall, maybe 2 (draft: 2.0); the city
"squished" north-south; 22 and 12 join the 25/11/3/4 row; 13, 23, 14, 15, 16, 20, 5 in one row;
southern rows shifted east toward the real map. The first layout is kept as `plan_2026_10_04`.

4b (push-apart tiles) is not built yet.

Kernel: **Chicago Council**.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

sys.path.insert(0, "../scripts")
import ward_maps
import build_ward_tiles

# --- Settings (every choice made in this notebook) --------------------------

TABLES_DIR = Path("../data/tables")

# Name on each tile: last name, same rule as ward_views ALDER_LABEL_NAME = "last"
# (text before the LAST comma of people.display_name).
TILE_LABEL_FONT_SIZE = 8
# Figure sizes for 1.3:1 tiles on a 9-column x 11-row grid (grid about as wide as tall).
FIGURE_SIZE_TILES = (11, 10)         # one tile map
FIGURE_SIZE_SIDE_BY_SIDE = (20, 10)  # real map | tiles

# Distances (§3) are measured in IL State Plane East (feet), not in Web Mercator, which stretches
# lengths by about 1.34x at Chicago's latitude. Converted to kilometers for display.
DISTANCE_CRS = "EPSG:3435"
KILOMETERS_PER_FOOT = 0.0003048

# Real-ward point that a tile is compared with (§3, §4): representative_point(), the same point
# ward labels use on the real map (ward_maps.draw_ward_labels). (Alternative: "centroid", which can
# fall outside an oddly-shaped ward.)
REAL_WARD_ANCHOR = "representative_point"

# §4 "neighbors" = two real wards whose boundaries share a line longer than this (meters,
# measured in DISTANCE_CRS and converted). 0 = any shared edge; corner-only contact never counts.
NEIGHBOR_MIN_SHARED_BOUNDARY_METERS = 0

# §6 tenure colors: same values as notebooks/ward_views.ipynb §2 settings
# (TENURE_COLORMAP, TENURE_COLOR_MIN_RULE = "data_min", TENURE_COLOR_CAP_YEARS).
TENURE_COLORMAP = "viridis_r"
TENURE_COLOR_CAP_YEARS = 22
TENURE_DECIMALS = 1

## 1. The layouts

Every layout in `build_ward_tiles.GRID_LAYOUTS`, as a table of grid cells (row 1 = north,
`·` = empty). Each row's start column is written out in the script; the layout in use is
`GRID_LAYOUT`.

In [ ]:
print("GRID_LAYOUT =", build_ward_tiles.GRID_LAYOUT, "  TILE_ASPECT =", build_ward_tiles.TILE_ASPECT,
      "  TILE_GAP =", build_ward_tiles.TILE_GAP, "  GRID_SIZE_FIT =", build_ward_tiles.GRID_SIZE_FIT)

def layout_table(layout_rows):
    layout_cells = build_ward_tiles.grid_cells(layout_rows)
    column_count = 1 + max(column for _, _, column in layout_cells)
    table = pd.DataFrame("·", index=range(1, len(layout_rows) + 1), columns=range(column_count), dtype=object)
    for ward, row_index, column_index in layout_cells:
        table.loc[row_index + 1, column_index] = ward
    table.insert(0, "start column", [start_column for start_column, _ in layout_rows])
    table.index.name = "row (1 = north)"
    return table

for layout_name, layout_rows in build_ward_tiles.GRID_LAYOUTS.items():
    print(f"\n{layout_name}" + ("   <- GRID_LAYOUT" if layout_name == build_ward_tiles.GRID_LAYOUT else ""))
    display(layout_table(layout_rows))

## 2. The tiles

Read back from `data/tables/ward_tiles_grid.geojson` through `ward_maps.load_ward_shapes("tiles_grid")`,
the same path every view will use. Labels: ward number and alder last name (current alder rule from
`ward_views` §1.1: `is_active` and a ward number in `elms_ward`).

In [ ]:
tiles = ward_maps.load_ward_shapes("tiles_grid")
real_wards = ward_maps.load_ward_shapes("real")
print(f"tiles: {len(tiles)}; distinct wards: {tiles.ward.nunique()}; same ward set as real map: "
      f"{set(tiles.ward) == set(real_wards.ward)}")

people_table = pd.read_csv(TABLES_DIR / "people.csv", dtype={"elms_ward": str})
has_ward_number = people_table.elms_ward.str.fullmatch(r"\d+", na=False)
current_alders = people_table[people_table.is_active & has_ward_number].copy()
current_alders["label_name"] = current_alders.display_name.str.rsplit(",", n=1).str[0].str.strip()
print(f"current alders: {len(current_alders)}")
last_name_by_ward = dict(zip(current_alders.elms_ward, current_alders.label_name))
tile_label_by_ward = {ward: f"{int(ward)}\n{last_name_by_ward[ward]}" for ward in tiles.ward}

def draw_tile_map(axis, tile_shapes, title, font_size=TILE_LABEL_FONT_SIZE):
    tile_shapes.plot(ax=axis, color="whitesmoke", edgecolor="none")
    ward_maps.draw_ward_outlines(axis, tile_shapes)
    annotations = ward_maps.draw_ward_labels(axis, tile_shapes, tile_label_by_ward, font_size=font_size)
    axis.set_title(title)
    axis.set_axis_off()
    axis.set_aspect("equal")
    return annotations

figure, axis = plt.subplots(figsize=FIGURE_SIZE_TILES)
annotations = draw_tile_map(axis, tiles, "Ward tiles (grid draft)")
overlaps = ward_maps.label_overlaps(figure, annotations)   # measured before show(), which releases the canvas
plt.show()
print(f"label pairs overlapping at {TILE_LABEL_FONT_SIZE} pt: {len(overlaps)}", overlaps)

## 3. Tiles vs the real map

Left: real wards. Right: tiles with the real ward outlines faintly behind, and a line from each
tile's center to its real ward's point (`REAL_WARD_ANCHOR`). The grid is placed by `GRID_SIZE_FIT`
and centered on the city's bounding box, so line lengths depend on that placement too.

In [ ]:
def real_anchor_points(ward_shapes):
    if REAL_WARD_ANCHOR == "representative_point":
        return ward_shapes.representative_point()
    if REAL_WARD_ANCHOR == "centroid":
        return ward_shapes.centroid
    raise ValueError(REAL_WARD_ANCHOR)

real_points = gpd.GeoSeries(real_anchor_points(real_wards).values, index=real_wards.ward, crs=real_wards.crs)
tile_centers = gpd.GeoSeries(tiles.centroid.values, index=tiles.ward, crs=tiles.crs)

figure, (real_axis, tile_axis) = plt.subplots(1, 2, figsize=FIGURE_SIZE_SIDE_BY_SIDE, sharex=True, sharey=True)
ward_maps.draw_ward_outlines(real_axis, real_wards)
ward_maps.draw_ward_labels(real_axis, real_wards, {ward: str(int(ward)) for ward in real_wards.ward}, font_size=8)
real_axis.set_title("Real wards (2023)")

real_wards.boundary.plot(ax=tile_axis, color="lightgray", linewidth=0.6)
ward_maps.draw_ward_outlines(tile_axis, tiles)
for ward in tiles.ward:
    tile_axis.plot([tile_centers[ward].x, real_points[ward].x], [tile_centers[ward].y, real_points[ward].y],
                   color="tab:red", linewidth=0.8)
real_points.plot(ax=tile_axis, color="tab:red", markersize=6)
ward_maps.draw_ward_labels(tile_axis, tiles, {ward: str(int(ward)) for ward in tiles.ward}, font_size=8)
tile_axis.set_title("Tiles; red line = to real ward point")
for axis in (real_axis, tile_axis):
    axis.set_axis_off()
    axis.set_aspect("equal")
plt.show()

In [ ]:
# Distance from each tile center to its real ward point, and the direction, in DISTANCE_CRS.
real_points_feet = real_points.to_crs(DISTANCE_CRS)
tile_centers_feet = tile_centers.to_crs(DISTANCE_CRS)
displacement = pd.DataFrame({
    "ward": tiles.ward.values,
    "alder": [last_name_by_ward[ward] for ward in tiles.ward],
    "east_km": [(tile_centers_feet[w].x - real_points_feet[w].x) * KILOMETERS_PER_FOOT for w in tiles.ward],
    "north_km": [(tile_centers_feet[w].y - real_points_feet[w].y) * KILOMETERS_PER_FOOT for w in tiles.ward],
})
displacement["distance_km"] = np.hypot(displacement.east_km, displacement.north_km)
print("tile center minus real ward point (positive east_km = tile is east of the real ward):")
display(displacement.sort_values("distance_km", ascending=False).round(2).reset_index(drop=True))
print(displacement.distance_km.describe().round(2))

**East-west alignment, independent of where the grid is placed.** For every ward: its real
east-west position (`REAL_WARD_ANCHOR`, km east of the westernmost ward point) against its grid
column, one panel per layout. Wards in the same grid row share a color; a row whose dots sit to
the left of the others is placed too far east on the grid for its real position, and the reverse.
The table lists the numbers drawn (this is what the southern start columns were set from).

In [ ]:
real_east_km = (real_points_feet.x - real_points_feet.x.min()) * KILOMETERS_PER_FOOT
real_north_km = (real_points_feet.y - real_points_feet.y.min()) * KILOMETERS_PER_FOOT
layout_names = list(build_ward_tiles.GRID_LAYOUTS)
figure, axes = plt.subplots(2, (len(layout_names) + 1) // 2, figsize=(16, 14), sharey=True)
for axis in axes.flat[len(layout_names):]:
    axis.set_axis_off()
alignment_tables = {}
for axis, layout_name in zip(axes.flat, layout_names):
    layout_cells = pd.DataFrame(build_ward_tiles.grid_cells(build_ward_tiles.GRID_LAYOUTS[layout_name]),
                                columns=["ward", "grid_row", "grid_column"])
    layout_cells["ward"] = layout_cells.ward.map("{:02d}".format)
    layout_cells["real_east_km"] = layout_cells.ward.map(real_east_km)
    layout_cells["real_north_km"] = layout_cells.ward.map(real_north_km)
    alignment_tables[layout_name] = layout_cells
    for row_index, row_cells in layout_cells.groupby("grid_row"):
        axis.scatter(row_cells.real_east_km, row_cells.grid_column, color=plt.cm.tab20(row_index % 20), s=30)
        for _, cell in row_cells.iterrows():
            axis.annotate(str(int(cell.ward)), (cell.real_east_km, cell.grid_column), xytext=(4, 2),
                          textcoords="offset points", fontsize=8)
    axis.set_xlabel("real east-west position (km east of westernmost ward point)")
    axis.set_ylabel("grid column")
    axis.set_title(layout_name)
    axis.grid(alpha=0.3)
plt.show()

for layout_name, layout_cells in alignment_tables.items():
    print(f"\n{layout_name}: each row's wards with (real east km, grid column)")
    for row_index, row_cells in layout_cells.groupby("grid_row"):
        print(f"  row {row_index + 1:2d}: " + "  ".join(
            f"{int(cell.ward)} ({cell.real_east_km:.1f}, col {cell.grid_column})" for _, cell in row_cells.iterrows()))

## 4. Neighbor order

For every pair of real wards sharing a boundary (`NEIGHBOR_MIN_SHARED_BOUNDARY_METERS`), compare
their direction on the real map with their direction on the grid. `east_order_flipped` = the ward
that is further east on the real map is further west on the grid (same idea north/south).
`real_east_km` / `real_north_km` show how far apart the pair really is, so a flip between two wards
that sit almost exactly north-south of each other (tiny `real_east_km`) can be told apart from a
large one. Grid ties (same column / same row) are reported as `tie`, not as a flip. Nothing is
filtered; every neighbor pair is listed.

In [ ]:
real_wards_feet = real_wards.to_crs(DISTANCE_CRS).set_index("ward")
min_shared_feet = NEIGHBOR_MIN_SHARED_BOUNDARY_METERS / 0.3048
tile_by_ward = tiles.set_index("ward")
grid_position = pd.DataFrame(build_ward_tiles.grid_cells(), columns=["ward", "grid_row", "grid_column"])
grid_position["ward"] = grid_position.ward.map("{:02d}".format)
grid_position = grid_position.set_index("ward")

def order_comparison(real_difference, grid_difference):
    if grid_difference == 0:
        return "tie"
    return "flipped" if np.sign(real_difference) != np.sign(grid_difference) else "same"

neighbor_rows = []
ward_list = sorted(real_wards_feet.index)
for position, ward_a in enumerate(ward_list):
    for ward_b in ward_list[position + 1:]:
        shared = real_wards_feet.geometry[ward_a].boundary.intersection(real_wards_feet.geometry[ward_b].boundary)
        if shared.length <= min_shared_feet:
            continue
        real_east = (real_points_feet[ward_b].x - real_points_feet[ward_a].x) * KILOMETERS_PER_FOOT
        real_north = (real_points_feet[ward_b].y - real_points_feet[ward_a].y) * KILOMETERS_PER_FOOT
        grid_east = grid_position.grid_column[ward_b] - grid_position.grid_column[ward_a]
        grid_north = grid_position.grid_row[ward_a] - grid_position.grid_row[ward_b]   # row 0 = north
        neighbor_rows.append({
            "ward_a": ward_a, "ward_b": ward_b, "shared_boundary_km": shared.length * KILOMETERS_PER_FOOT,
            "real_east_km": real_east, "real_north_km": real_north,
            "grid_columns_east": grid_east, "grid_rows_north": grid_north,
            "east_order": order_comparison(real_east, grid_east),
            "north_order": order_comparison(real_north, grid_north),
            "tiles_touch_or_diagonal": max(abs(grid_east), abs(grid_north)) == 1,
        })
neighbors = pd.DataFrame(neighbor_rows)
print(f"real neighbor pairs: {len(neighbors)}")
print("east_order:", neighbors.east_order.value_counts().to_dict(), "  north_order:", neighbors.north_order.value_counts().to_dict())
print(f"real neighbors that are NOT adjacent on the grid (incl. diagonal): {(~neighbors.tiles_touch_or_diagonal).sum()}")
print("\npairs with any flip:")
display(neighbors[(neighbors.east_order == "flipped") | (neighbors.north_order == "flipped")].round(2))
print("real neighbors not adjacent on the grid:")
display(neighbors[~neighbors.tiles_touch_or_diagonal].round(2))

## 5. Options

Each panel is built in memory with one setting changed from `scripts/build_ward_tiles.py`
(nothing is written). To adopt one, change the constant in the script and re-run
`build_ward_tiles.py grid`.

**5.1 Layouts.** Every entry of `GRID_LAYOUTS`, with the script's `TILE_ASPECT`.

In [ ]:
layout_names = list(build_ward_tiles.GRID_LAYOUTS)
figure, axes = plt.subplots(2, (len(layout_names) + 1) // 2, figsize=(22, 20))
for axis in axes.flat[len(layout_names):]:
    axis.set_axis_off()
for axis, layout_name in zip(axes.flat, layout_names):
    option_tiles = build_ward_tiles.build_grid_tiles(layout_rows=build_ward_tiles.GRID_LAYOUTS[layout_name])
    draw_tile_map(axis, option_tiles, layout_name, font_size=7)
plt.show()

**5.2 Southern start columns.** The four southern rows' start columns are a draft (OPEN QUESTION in
the script). Each panel moves ONE of those rows one column west; everything else as in `GRID_LAYOUT`.

In [ ]:
base_layout = build_ward_tiles.GRID_LAYOUTS[build_ward_tiles.GRID_LAYOUT]
SOUTHERN_ROW_INDEXES = [len(base_layout) - 4, len(base_layout) - 3, len(base_layout) - 2, len(base_layout) - 1]
panels = [("as in GRID_LAYOUT", base_layout)]
for row_index in SOUTHERN_ROW_INDEXES:
    start_column, row = base_layout[row_index]
    shifted_layout = list(base_layout)
    shifted_layout[row_index] = (start_column - 1, row)
    first_ward = next(ward for ward in row if ward is not None)
    panels.append((f"row {row_index + 1} ({first_ward} ...) start {start_column} -> {start_column - 1}", shifted_layout))

figure, axes = plt.subplots(len(panels), 1, figsize=(12, 8 * len(panels)))
for axis, (title, layout_rows) in zip(axes, panels):
    try:
        option_tiles = build_ward_tiles.build_grid_tiles(layout_rows=layout_rows)
    except AssertionError as error:   # e.g. a shift that puts two wards in the same cell
        axis.set_title(f"{title}: not possible ({error})")
        axis.set_axis_off()
        continue
    draw_tile_map(axis, option_tiles, title, font_size=7)
plt.show()

**5.3 Tile aspect** (user, 2026-10-05: 1.3) and **gap**.

In [ ]:
aspect_gap_options = [(1.3, 0.0), (1.3, 0.08), (1.3, 0.15), (1.5, 0.08), (2.0, 0.08)]
figure, axes = plt.subplots(len(aspect_gap_options), 1, figsize=(12, 7 * len(aspect_gap_options)))
for axis, (aspect, gap) in zip(axes, aspect_gap_options):
    option_tiles = build_ward_tiles.build_grid_tiles(tile_aspect=aspect, tile_gap=gap)
    draw_tile_map(axis, option_tiles, f"TILE_ASPECT {aspect}, TILE_GAP {gap}", font_size=7)
plt.show()

**5.4 Size fit** (only changes where the grid sits over the real outlines).

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_SIDE_BY_SIDE)
for axis, fit in zip(axes, ["city_width", "city_height"]):
    option_tiles = build_ward_tiles.build_grid_tiles(grid_size_fit=fit)
    real_wards.boundary.plot(ax=axis, color="lightgray", linewidth=0.6)
    draw_tile_map(axis, option_tiles, f"GRID_SIZE_FIT {fit}", font_size=6)
plt.show()

## 6. A view on tiles: tenure

`ward_views` §2's tenure fill (`people.years_on_council`, `viridis_r`, lowest value to 22 years),
drawn on `MAP_SHAPES = "real"`, `"tiles_grid"` and `"tiles_pushed"` with the same helper (`ward_maps.draw_ward_fill`).
Shows the switch works; the real-map version in `ward_views` is unchanged.

In [ ]:
current_alders["ward"] = current_alders.elms_ward
color_min = current_alders.years_on_council.min()   # TENURE_COLOR_MIN_RULE = "data_min"
figure, axes = plt.subplots(1, 3, figsize=(24, 11))
for axis, map_shapes in zip(axes, ["real", "tiles_grid", "tiles_pushed"]):
    shapes = ward_maps.load_ward_shapes(map_shapes).merge(current_alders[["ward", "years_on_council"]], on="ward")
    color_scale = ward_maps.draw_ward_fill(axis, shapes, "years_on_council", TENURE_COLORMAP,
                                           color_min, TENURE_COLOR_CAP_YEARS, opacity=0.85)
    ward_maps.draw_ward_outlines(axis, shapes)
    labels = {ward: f"{int(ward)}\n{years:.{TENURE_DECIMALS}f}" for ward, years in zip(shapes.ward, shapes.years_on_council)}
    ward_maps.draw_ward_labels(axis, shapes, labels, font_size=7)
    axis.set_title(f"Years on council — MAP_SHAPES = {map_shapes!r}")
    axis.set_axis_off()
    axis.set_aspect("equal")
figure.colorbar(color_scale, ax=axes, shrink=0.5, label="years on council")
plt.show()

## 7. Push-apart ("explode") tiles — plan 4b

`scripts/build_ward_tiles.py pushed` → `data/tables/ward_tiles_pushed.geojson`. No grid: each
ward's tile starts at its real ward point (`PUSHED_ANCHOR_METHOD`), the start points are squished
north-south (`PUSHED_VERTICAL_SCALE`), every tile gets the same size (`PUSHED_TILE_SIZE_FACTOR` ×
median nearest-neighbor distance, `PUSHED_TILE_ASPECT`), and overlapping pairs are pushed apart
round by round until none overlap (`PUSH_*` settings; every one is described in the script).
`PUSH_MIN_STEP_FRACTION` was added 2026-10-05: without it, tiles squeezed from both sides never
quite finished separating, for every size above 1.0.

In [ ]:
print({name: getattr(build_ward_tiles, name) for name in dir(build_ward_tiles)
       if name.startswith(("PUSHED_", "PUSH_"))})
pushed_tiles = ward_maps.load_ward_shapes("tiles_pushed")
pushed_detail = build_ward_tiles.build_pushed_tiles()   # same settings, kept in memory for start points / rounds
print(f"tiles: {len(pushed_tiles)}; push rounds: {pushed_detail.rounds.iloc[0]}; "
      f"tile width {pushed_detail.tile_width.iloc[0] / 1000:.2f} km x height {pushed_detail.tile_height.iloc[0] / 1000:.2f} km "
      "(Web Mercator meters, about 1.34x ground distance)")

figure, axes = plt.subplots(1, 3, figsize=(24, 11))
real_axis, grid_axis, pushed_axis = axes
ward_maps.draw_ward_outlines(real_axis, real_wards)
ward_maps.draw_ward_labels(real_axis, real_wards, {ward: str(int(ward)) for ward in real_wards.ward}, font_size=8)
real_axis.set_title("Real wards (2023)")
draw_tile_map(grid_axis, tiles, f"Grid tiles ({build_ward_tiles.GRID_LAYOUT})", font_size=7)
draw_tile_map(pushed_axis, pushed_tiles, "Pushed tiles", font_size=7)
for axis in axes:
    axis.set_axis_off()
    axis.set_aspect("equal")
plt.show()

**Where each pushed tile ended up.** Light gray: real ward outlines. Red dot: the tile's start point
(real point after any hand move and the north-south squish). Red line: start point → final tile center. The table
lists each tile's move, largest first.

In [ ]:
figure, axis = plt.subplots(figsize=(11, 13))
real_wards.boundary.plot(ax=axis, color="lightgray", linewidth=0.6)
ward_maps.draw_ward_outlines(axis, pushed_tiles)
pushed_centers = pushed_detail.set_index("ward").geometry.centroid.to_crs(ward_maps.PLOT_CRS)
pushed_starts = gpd.GeoSeries(gpd.points_from_xy(pushed_detail.start_x, pushed_detail.start_y),
                              index=pushed_detail.ward, crs=build_ward_tiles.TILE_BUILD_CRS).to_crs(ward_maps.PLOT_CRS)
for ward in pushed_detail.ward:
    axis.plot([pushed_starts[ward].x, pushed_centers[ward].x], [pushed_starts[ward].y, pushed_centers[ward].y],
              color="tab:red", linewidth=0.8)
pushed_starts.plot(ax=axis, color="tab:red", markersize=6)
ward_maps.draw_ward_labels(axis, pushed_tiles, {ward: str(int(ward)) for ward in pushed_tiles.ward}, font_size=7)
axis.set_title("Pushed tiles: start point (red dot) -> final position")
axis.set_axis_off()
axis.set_aspect("equal")
plt.show()

pushed_moves = pd.DataFrame({
    "ward": pushed_detail.ward,
    "alder": pushed_detail.ward.map(last_name_by_ward),
    "east_km": (pushed_centers.to_crs(DISTANCE_CRS).x.values - pushed_starts.to_crs(DISTANCE_CRS).x.values) * KILOMETERS_PER_FOOT,
    "north_km": (pushed_centers.to_crs(DISTANCE_CRS).y.values - pushed_starts.to_crs(DISTANCE_CRS).y.values) * KILOMETERS_PER_FOOT,
})
pushed_moves["distance_km"] = np.hypot(pushed_moves.east_km, pushed_moves.north_km)
print("final tile center minus start point (after squish), ground km:")
display(pushed_moves.sort_values("distance_km", ascending=False).round(2).reset_index(drop=True))
print(pushed_moves.distance_km.describe().round(2))

**Neighbor order, pushed vs grid.** Same idea as §4, but comparing tile centers instead of grid
rows/columns, so both tile sets are measured the same way. For every pair of real neighbors:
is the pair's east-west order (and north-south order) the same on the tiles as on the real map?
`tie` = the two tile centers are within `CENTER_TIE_FRACTION` of a tile width (east-west) or
height (north-south) of each other, e.g. two grid tiles in the same column.

In [ ]:
CENTER_TIE_FRACTION = 0.25   # judgment call: centers closer than a quarter tile count as "level"

def tile_neighbor_orders(tile_shapes):
    centers = tile_shapes.set_index("ward").geometry.centroid
    bounds = tile_shapes.geometry.iloc[0].bounds
    tile_width, tile_height = bounds[2] - bounds[0], bounds[3] - bounds[1]
    def compare(real_difference, tile_difference, tile_size):
        if abs(tile_difference) < CENTER_TIE_FRACTION * tile_size:
            return "tie"
        return "flipped" if np.sign(real_difference) != np.sign(tile_difference) else "same"
    rows = []
    for _, pair in neighbors.iterrows():
        tile_east = centers[pair.ward_b].x - centers[pair.ward_a].x
        tile_north = centers[pair.ward_b].y - centers[pair.ward_a].y
        rows.append({"ward_a": pair.ward_a, "ward_b": pair.ward_b,
                     "east_order": compare(pair.real_east_km, tile_east, tile_width),
                     "north_order": compare(pair.real_north_km, tile_north, tile_height),
                     "tiles_touch": tile_shapes.set_index("ward").geometry[pair.ward_a].buffer(
                         0.2 * min(tile_width, tile_height)).intersects(tile_shapes.set_index("ward").geometry[pair.ward_b])})
    return pd.DataFrame(rows)

orders_by_tiles = {"grid": tile_neighbor_orders(tiles), "pushed": tile_neighbor_orders(pushed_tiles)}
summary = pd.DataFrame({
    name: {**{f"east {key}": value for key, value in table.east_order.value_counts().items()},
           **{f"north {key}": value for key, value in table.north_order.value_counts().items()},
           "real neighbors whose tiles are within 0.2 tile of touching": int(table.tiles_touch.sum())}
    for name, table in orders_by_tiles.items()}).fillna(0).astype(int)
print(f"real neighbor pairs: {len(neighbors)}")
display(summary)
for name, table in orders_by_tiles.items():
    print(f"\n{name}: pairs with a flip")
    display(table[(table.east_order == "flipped") | (table.north_order == "flipped")])

**Options.** Built in memory with one or two settings changed from the script; nothing is written.
(1) Tile size × gap at the script's squish. (2) `PUSHED_TILE_ASPECT` 1.5 vs 2.0, in case "1.5"
meant the shape. (3) Ward 41's hand move east: 0, 3, 5, 7 km. (4) Size × squish, for reference.

In [ ]:
def draw_options(options, column_count, panel_size, title_of):
    row_count = (len(options) + column_count - 1) // column_count
    figure, axes = plt.subplots(row_count, column_count, figsize=(panel_size[0] * column_count, panel_size[1] * row_count))
    for axis in np.atleast_1d(axes).flat[len(options):]:
        axis.set_axis_off()
    for axis, option in zip(np.atleast_1d(axes).flat, options):
        option_tiles = build_ward_tiles.build_pushed_tiles(**option)
        draw_tile_map(axis, option_tiles, f"{title_of(option)}, rounds {option_tiles.rounds.iloc[0]}", font_size=6)
    plt.show()

draw_options([{"tile_size_factor": size, "tile_gap": gap} for gap in (0.08, 0.20, 0.30) for size in (1.5, 1.8)],
             2, (10, 11), lambda option: f"size {option['tile_size_factor']}, gap {option['tile_gap']}")
draw_options([{"tile_aspect": aspect} for aspect in (1.5, 2.0)],
             2, (10, 11), lambda option: f"PUSHED_TILE_ASPECT {option['tile_aspect']}")
draw_options([{"start_shift_km_by_ward": {"41": (east_km, 0.0)}} for east_km in (0, 3, 5, 7)],
             2, (10, 11), lambda option: f"41 moved {option['start_shift_km_by_ward']['41'][0]} km east")
draw_options([{"tile_size_factor": size, "vertical_scale": squish} for squish in (1.0, 0.9, 0.75) for size in (1.2, 1.5, 1.8)],
             3, (8, 9), lambda option: f"size {option['tile_size_factor']}, squish {option['vertical_scale']}")